# 03 — Sepsis Primary Analysis (DR-RF)

Loads experiment results from the checkpoint and generates the **primary manuscript
tables and figures** using DR-RF only.

Sections:
1. Load results
2. Primary results table (ATE MSE + CATE MSE, mean ± 95 % CI)
3. Primary Bar Chart
4. Paired-difference forest plot (CDV_SEPARATE vs each method)
5. CDV support summary
6. Overlap diagnostics
7. Export primary tables

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
REVISED_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
for p in [PROJECT_ROOT, REVISED_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(REVISED_ROOT, 'sepsis'))
import config as CFG

from helpers.runner import load_checkpoint
from helpers.metrics import (
    build_primary_table, build_full_learner_table,
    paired_ci, format_ci, resolve_alternative, describe_paired_test,
    build_paper_summary_table, significance_stars,
    METHODS_ORDER, LEARNERS_ORDER
)
from helpers.plotting import (
    plot_primary_bar_table, plot_cdv_support_heatmap, METHOD_LABELS
)

os.makedirs(CFG.PLOTS_DIR, exist_ok=True)
print(f'Setup complete. CI_METHOD={CFG.CI_METHOD!r}, CI_SIDED={CFG.CI_SIDED!r}')


## 1. Load Results

In [ ]:
results_by_seed = load_checkpoint(CFG.CHECKPOINT_PATH)
completed_seeds = sorted(results_by_seed.keys())
print(f'Loaded {len(completed_seeds)} seeds: {completed_seeds}')

if len(completed_seeds) < 5:
    print('WARNING: fewer than 5 seeds. Results will be unreliable.')

## 2. Primary ATE MSE Table (DR-RF)

In [ ]:
primary_table = build_primary_table(
    results_by_seed,
    learner=CFG.PRIMARY_LEARNER,
    ci_method=CFG.CI_METHOD,
    sided=CFG.CI_SIDED,
)
print(f'Primary Results Table ({CFG.PRIMARY_LEARNER}) — ATE MSE + CATE MSE')
print(describe_paired_test('method', 'GLOBAL_SENTINEL', 'ATE MSE', lower_is_better=True,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))
print(describe_paired_test('method', 'GLOBAL_SENTINEL', 'CATE MSE', lower_is_better=True,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))
display(primary_table)


## 3. Primary Bar Chart (DR-RF)

In [ ]:
fig = plot_primary_bar_table(
    results_by_seed,
    learner=CFG.PRIMARY_LEARNER,
    metric='ate_mse',
    title=f'ATE MSE by Method ({CFG.PRIMARY_LEARNER}) — Sepsis',
    save_path=os.path.join(CFG.PLOTS_DIR, 'primary_ate_mse.png'),
)
plt.show()

fig2 = plot_primary_bar_table(
    results_by_seed,
    learner=CFG.PRIMARY_LEARNER,
    metric='cate_mse',
    title=f'CATE MSE by Method ({CFG.PRIMARY_LEARNER}) — Sepsis',
    save_path=os.path.join(CFG.PLOTS_DIR, 'primary_cate_mse.png'),
)
plt.show()

## 3b. Primary Bar Chart — Kendall Tau & Spearman Rho (DR-RF)

In [ ]:
fig3 = plot_primary_bar_table(
    results_by_seed,
    learner=CFG.PRIMARY_LEARNER,
    metric='kendall_tau',
    title=f'Kendall Tau by Method ({CFG.PRIMARY_LEARNER}) — Sepsis',
    save_path=os.path.join(CFG.PLOTS_DIR, 'primary_kendall_tau.png'),
)
plt.show()

fig4 = plot_primary_bar_table(
    results_by_seed,
    learner=CFG.PRIMARY_LEARNER,
    metric='spearman_rho',
    title=f'Spearman Rho by Method ({CFG.PRIMARY_LEARNER}) — Sepsis',
    save_path=os.path.join(CFG.PLOTS_DIR, 'primary_spearman_rho.png'),
)
plt.show()

## 4. Paired Differences — CDV_SEPARATE vs Each Method

In [ ]:
alt_ate = resolve_alternative(CFG.CI_SIDED, lower_is_better=True)
print(describe_paired_test('CDV_SEPARATE', 'method', 'ATE MSE', lower_is_better=True,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))

cdv_vals = np.array([
    sr['metrics'].get('CDV_SEPARATE', {}).get(CFG.PRIMARY_LEARNER, {}).get('ate_mse', np.nan)
    for sr in results_by_seed.values()
])

paired_ate_rows = [{
    'method': METHOD_LABELS.get('CDV_SEPARATE', 'CDV_SEPARATE'),
    'mean_ate_mse': np.nanmean(cdv_vals),
    'std_ate_mse': np.nanstd(cdv_vals),
    'paired 95% CI of Δ (CDV_SEPARATE − method)': '-',
    'p_value (paired)': '-',
}]
for method in METHODS_ORDER:
    if method == 'CDV_SEPARATE':
        continue
    other_vals = np.array([
        sr['metrics'].get(method, {}).get(CFG.PRIMARY_LEARNER, {}).get('ate_mse', np.nan)
        for sr in results_by_seed.values()
    ])
    res = paired_ci(cdv_vals, other_vals, method=CFG.CI_METHOD, alternative=alt_ate)
    paired_ate_rows.append({
        'method': METHOD_LABELS.get(method, method),
        'mean_ate_mse': np.nanmean(other_vals),
        'std_ate_mse': np.nanstd(other_vals),
        'paired 95% CI of Δ (CDV_SEPARATE − method)': format_ci(res['ci_lo'], res['ci_hi']),
        'p_value (paired)': res['p_value'],
    })

paired_ate_table = pd.DataFrame(paired_ate_rows)
display(paired_ate_table)


## 4a. Paired Differences — CDV_SEPARATE vs Each Method (CATE MSE)


In [ ]:
alt_cate = resolve_alternative(CFG.CI_SIDED, lower_is_better=True)
print(describe_paired_test('CDV_SEPARATE', 'method', 'CATE MSE', lower_is_better=True,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))

cdv_cate_vals = np.array([
    sr['metrics'].get('CDV_SEPARATE', {}).get(CFG.PRIMARY_LEARNER, {}).get('cate_mse', np.nan)
    for sr in results_by_seed.values()
])

paired_cate_rows = [{
    'method': METHOD_LABELS.get('CDV_SEPARATE', 'CDV_SEPARATE'),
    'mean_cate_mse': np.nanmean(cdv_cate_vals),
    'std_cate_mse': np.nanstd(cdv_cate_vals),
    'paired 95% CI of Δ (CDV_SEPARATE − method)': '-',
    'p_value (paired)': '-',
}]
for method in METHODS_ORDER:
    if method == 'CDV_SEPARATE':
        continue
    other_cate_vals = np.array([
        sr['metrics'].get(method, {}).get(CFG.PRIMARY_LEARNER, {}).get('cate_mse', np.nan)
        for sr in results_by_seed.values()
    ])
    res = paired_ci(cdv_cate_vals, other_cate_vals, method=CFG.CI_METHOD, alternative=alt_cate)
    paired_cate_rows.append({
        'method': METHOD_LABELS.get(method, method),
        'mean_cate_mse': np.nanmean(other_cate_vals),
        'std_cate_mse': np.nanstd(other_cate_vals),
        'paired 95% CI of Δ (CDV_SEPARATE − method)': format_ci(res['ci_lo'], res['ci_hi']),
        'p_value (paired)': res['p_value'],
    })

paired_cate_table = pd.DataFrame(paired_cate_rows)
display(paired_cate_table)


## 4b. CDV_SEPARATE — Per-Group Metric Breakdown

Breaks ATE MSE and CATE MSE by CDV subgroup (CDV_1, CDV_2, OTHER) to identify
which subgroup drives degradation.  A CDV with high error vs. GLOBAL_SENTINEL
(on the same test subset) is the smoking gun.

In [ ]:
from helpers.metrics import ate_mse, cate_mse

LEARNER = CFG.PRIMARY_LEARNER
breakdown_rows = []

for seed, sr in sorted(results_by_seed.items()):
    cdv_pred = sr['predictions'].get('CDV_SEPARATE', {}).get(LEARNER, {})
    glob_pred = sr['predictions'].get('GLOBAL_SENTINEL', {}).get(LEARNER, {})
    ite_pred = cdv_pred.get('ite_pred', np.array([]))
    ite_true = cdv_pred.get('ite_true', np.array([]))
    variant = cdv_pred.get('variant', np.array([]))

    if len(ite_pred) == 0:
        continue

    retained = sr.get('retained_cdv_info', {})
    g_pred = glob_pred.get('ite_pred', np.array([]))
    g_true = glob_pred.get('ite_true', np.array([]))

    for group in sorted(set(variant)):
        mask = variant == group
        n_treated_train = retained.get(group, {}).get('n_treated', np.nan)
        breakdown_rows.append({
            'seed': seed,
            'group': group,
            'n_test': mask.sum(),
            'n_treated_train': n_treated_train,
            'cdv_ate_mse': ate_mse(ite_pred[mask], ite_true[mask]),
            'cdv_cate_mse': cate_mse(ite_pred[mask], ite_true[mask]),
            'global_ate_mse': ate_mse(g_pred[mask], g_true[mask]) if len(g_pred) else np.nan,
            'global_cate_mse': cate_mse(g_pred[mask], g_true[mask]) if len(g_pred) else np.nan,
        })

per_group_breakdown = pd.DataFrame(breakdown_rows)
display(per_group_breakdown)

## 5. CDV Support Summary

In [ ]:
# Aggregate per-CDV support statistics across seeds
all_support = []
for seed, sr in results_by_seed.items():
    ss = sr.get('support_stats')
    if isinstance(ss, pd.DataFrame) and not ss.empty:
        ss = ss.copy()
        ss['outer_seed'] = seed
        all_support.append(ss)

if all_support:
    support_df = pd.concat(all_support, ignore_index=True)
    summary = support_df.groupby('cdv_id').agg(
        n_mean=('n', 'mean'), n_std=('n', 'std'),
        n_treated_mean=('n_treated', 'mean'),
        treatment_rate_mean=('treatment_rate', 'mean'),
        overlap_frac_mean=('overlap_fraction', 'mean'),
        ess_mean=('ess', 'mean'),
        seeds_present=('outer_seed', 'nunique'),
    ).round(3)
    print('CDV Support Summary (mean across seeds):')
    display(summary)
    support_df.to_parquet(os.path.join(CFG.ARTIFACTS_DIR, 'support_stats.parquet'), index=False)
    print(f'Saved to: {CFG.ARTIFACTS_DIR}/support_stats.parquet')
else:
    print('No support stats available.')

## 6. Overlap Diagnostics

In [ ]:
if all_support:
    pct_outside = support_df.groupby('cdv_id')['pct_outside_overlap'].mean()
    overlap_summary = pct_outside.round(2).to_frame('pct_outside_overlap_mean')
    display(overlap_summary)

violation_counts = {
    seed: len(sr.get('violations_log', {}))
    for seed, sr in results_by_seed.items()
}
violation_table = (
    pd.Series(violation_counts, name='n_cdv_violations')
    .rename_axis('seed')
    .reset_index()
)
violation_summary = pd.DataFrame([{
    'seeds_with_violations': (violation_table['n_cdv_violations'] > 0).sum(),
    'total_seeds': len(violation_table),
}])
display(violation_table)
display(violation_summary)

## 7. Export Primary Tables

In [ ]:
primary_table.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'primary_results_table.csv'), index=False)
print('Table exported to artifacts/.')

## 8. Paper Summary Table — ATE & CATE MSE

Row index: (Metric, Method) — CDV_SEPARATE is the top row of each metric block
(reference, own raw mean ± std, no CI/significance). Every other method shows
its own raw mean ± std plus the paired 95% CI and significance of
Δ = CDV_SEPARATE − method (see `CFG.CI_METHOD` / `CFG.CI_SIDED`).
Significance: `*` p<0.1, `**` p<0.05, `***` p<0.01, `-` not significant.

In [ ]:
print(describe_paired_test('CDV_SEPARATE', 'method', 'ATE/CATE MSE', lower_is_better=True,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))

ate_cate_summary = build_paper_summary_table(
    results_by_seed,
    learner=CFG.PRIMARY_LEARNER,
    metrics=['ate_mse', 'cate_mse'],
    metric_labels={'ate_mse': 'ATE MSE', 'cate_mse': 'CATE MSE'},
    method_labels=METHOD_LABELS,
    lower_is_better={'ate_mse': True, 'cate_mse': True},
    ci_method=CFG.CI_METHOD,
    sided=CFG.CI_SIDED,
)
display(ate_cate_summary)
ate_cate_summary.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'paper_summary_ate_cate.csv'))


## 9. Paper Summary Table — Kendall τ & Spearman ρ

Same layout as Section 8, but the primary row key is the ranking metric
(Kendall τ / Spearman ρ, higher is better) and the "Mean ± Std" column shows
each method's own rank-correlation mean ± std (instead of MSE).


In [ ]:
print(describe_paired_test('CDV_SEPARATE', 'method', 'Kendall τ / Spearman ρ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD))

rank_summary = build_paper_summary_table(
    results_by_seed,
    learner=CFG.PRIMARY_LEARNER,
    metrics=['kendall_tau', 'spearman_rho'],
    metric_labels={'kendall_tau': 'Kendall τ', 'spearman_rho': 'Spearman ρ'},
    method_labels=METHOD_LABELS,
    lower_is_better={'kendall_tau': False, 'spearman_rho': False},
    ci_method=CFG.CI_METHOD,
    sided=CFG.CI_SIDED,
)
display(rank_summary)
rank_summary.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'paper_summary_kendall_spearman.csv'))


## 10. Compact Signed Summary Tables — ATE/CATE MSE and Kendall τ/Spearman ρ

Two compact tables for the main text (kept separate so error and ranking
metrics never share a paired-test family): rows = method (CDV_SEPARATE
first, reference); columns grouped by metric, each with two sub-columns:
raw **Mean ± Std** and a **signed Δ%** vs CDV_SEPARATE (positive always
means CDV_SEPARATE is better, regardless of metric direction) with
significance stars from a two-sided paired test (see `CFG.CI_METHOD`). This
avoids the earlier issue of a two-sided p-value/CI not directly indicating
*which* method is better.
Significance: `*` p<0.1, `**` p<0.05, `***` p<0.01, `-` not significant.

In [ ]:
from helpers.metrics import build_signed_summary_table

print(describe_paired_test('CDV_SEPARATE', 'method', 'ATE/CATE MSE',
                            lower_is_better=True, sided='two-sided', ci_method=CFG.CI_METHOD))

signed_summary_ate_cate = build_signed_summary_table(
    results_by_seed,
    learner=CFG.PRIMARY_LEARNER,
    metrics=['ate_mse', 'cate_mse'],
    metric_labels={'ate_mse': 'ATE MSE', 'cate_mse': 'CATE MSE'},
    method_labels=METHOD_LABELS,
    lower_is_better={'ate_mse': True, 'cate_mse': True},
    ci_method=CFG.CI_METHOD,
    sided='two-sided',
)
display(signed_summary_ate_cate)
signed_summary_ate_cate.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'signed_summary_ate_cate.csv'))

print(describe_paired_test('CDV_SEPARATE', 'method', 'Kendall τ / Spearman ρ',
                            lower_is_better=False, sided='two-sided', ci_method=CFG.CI_METHOD))

signed_summary_rank = build_signed_summary_table(
    results_by_seed,
    learner=CFG.PRIMARY_LEARNER,
    metrics=['kendall_tau', 'spearman_rho'],
    metric_labels={'kendall_tau': 'Kendall τ', 'spearman_rho': 'Spearman ρ'},
    method_labels=METHOD_LABELS,
    lower_is_better={'kendall_tau': False, 'spearman_rho': False},
    ci_method=CFG.CI_METHOD,
    sided='two-sided',
)
display(signed_summary_rank)
signed_summary_rank.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'signed_summary_kendall_spearman.csv'))


## 11. Multiplicity-Adjusted Summary Tables (FDR-BH correction)

Sections 8–10 each report a **raw, unadjusted** paired p-value per (metric,
method) cell — 4 non-reference methods × up to 2 metrics per table, i.e.
several simultaneous pairwise comparisons — so the raw significance stars
there are optimistic (inflated false-positive rate under multiple testing).
The tables below repeat 8/9/10 with p-values adjusted for multiplicity
across all comparisons **within each table**, using Benjamini-Hochberg
(`adjust_method='fdr_bh'`, controls the false discovery rate; see
`helpers.metrics.adjust_pvalues`). Error metrics (ATE/CATE MSE) and ranking
metrics (Kendall τ/Spearman ρ) form separate FDR families, never pooled
together. Only the significance stars change here — means, CIs, and Δ% are
identical to the unadjusted tables above.

In [ ]:
print(describe_paired_test('CDV_SEPARATE', 'method', 'ATE/CATE MSE', lower_is_better=True,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD) + '  [stars: FDR-BH adjusted across table]')

ate_cate_summary_fdr = build_paper_summary_table(
    results_by_seed,
    learner=CFG.PRIMARY_LEARNER,
    metrics=['ate_mse', 'cate_mse'],
    metric_labels={'ate_mse': 'ATE MSE', 'cate_mse': 'CATE MSE'},
    method_labels=METHOD_LABELS,
    lower_is_better={'ate_mse': True, 'cate_mse': True},
    ci_method=CFG.CI_METHOD,
    sided=CFG.CI_SIDED,
    adjust_method='fdr_bh',
)
display(ate_cate_summary_fdr)
ate_cate_summary_fdr.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'paper_summary_ate_cate_fdr_bh.csv'))


In [ ]:
print(describe_paired_test('CDV_SEPARATE', 'method', 'Kendall τ / Spearman ρ', lower_is_better=False,
                            sided=CFG.CI_SIDED, ci_method=CFG.CI_METHOD) + '  [stars: FDR-BH adjusted across table]')

rank_summary_fdr = build_paper_summary_table(
    results_by_seed,
    learner=CFG.PRIMARY_LEARNER,
    metrics=['kendall_tau', 'spearman_rho'],
    metric_labels={'kendall_tau': 'Kendall τ', 'spearman_rho': 'Spearman ρ'},
    method_labels=METHOD_LABELS,
    lower_is_better={'kendall_tau': False, 'spearman_rho': False},
    ci_method=CFG.CI_METHOD,
    sided=CFG.CI_SIDED,
    adjust_method='fdr_bh',
)
display(rank_summary_fdr)
rank_summary_fdr.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'paper_summary_kendall_spearman_fdr_bh.csv'))


In [ ]:
from helpers.metrics import build_signed_summary_table

print(describe_paired_test('CDV_SEPARATE', 'method', 'ATE/CATE MSE / Kendall τ / Spearman ρ',
                            lower_is_better=True, sided='two-sided', ci_method=CFG.CI_METHOD)
      + '  [stars: FDR-BH adjusted, 3 families: ATE MSE alone, CATE MSE alone, Kendall τ + Spearman ρ together]')

signed_summary_fdr = build_signed_summary_table(
    results_by_seed,
    learner=CFG.PRIMARY_LEARNER,
    metrics=['ate_mse', 'cate_mse', 'kendall_tau', 'spearman_rho'],
    metric_labels={'ate_mse': 'ATE MSE', 'cate_mse': 'CATE MSE',
                   'kendall_tau': 'Kendall τ', 'spearman_rho': 'Spearman ρ'},
    method_labels=METHOD_LABELS,
    lower_is_better={'ate_mse': True, 'cate_mse': True, 'kendall_tau': False, 'spearman_rho': False},
    ci_method=CFG.CI_METHOD,
    sided='two-sided',
    adjust_method='fdr_bh',
    metric_families={'kendall_tau': 'RANK', 'spearman_rho': 'RANK'},
)
display(signed_summary_fdr)
signed_summary_fdr.to_csv(os.path.join(CFG.ARTIFACTS_DIR, 'signed_summary_ate_cate_kendall_spearman_fdr_bh.csv'))
